In [2]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
import re
import gc
import math
import torch
import numpy as np
import evaluate
from datasets import load_from_disk, DatasetDict
from transformers import (
    WhisperProcessor, 
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model 
from typing import Dict, List, Union

/opt/miniconda3/envs/whisper_fine_tuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_PATH = "Processed_Hindi_Parquet_Dataset"
MODEL_ID = "openai/whisper-small" 
LANGUAGE = "hindi"
TASK = "transcribe" 
OUTPUT_DIR = "whisper-small-lora-ap"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS = 10
LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
EVAL_STEPS = 500
SAVE_STEPS = 500

LORA_CFG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)

PUNCTUATION_TO_REMOVE = r"[.,?!।:;\"'()\[\]]"
NOISE_MARKERS = r"<noise>|<laughter>|<music>|\[\w+\]|\(\w+\)"

#metrics
metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

processor = None
feature_extractor = None
tokenizer = None
sampling_rate = None

#functions to clean data
def clean_transcript(text):
    if not isinstance(text, str):
        return "" 
    text = re.sub(NOISE_MARKERS, "", text)
    text = re.sub(PUNCTUATION_TO_REMOVE, "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def prepare(batch):
    cleaned_text = clean_transcript(batch["text"])
    audio = batch["audio"]
    
    inputs = feature_extractor(
        audio["array"], 
        sampling_rate=sampling_rate, 
        return_tensors="pt"
    )
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = tokenizer(cleaned_text).input_ids
    
    return batch

#data collator
class WhisperDataCollator:
    def __init__(self, feature_extractor, tokenizer):
        self.feature_extractor = feature_extractor
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_features = [b["input_features"] for b in batch]
        batch_inputs = self.feature_extractor.pad(
            [{"input_features": f} for f in input_features],
            return_tensors="pt"
        )

        labels = [b["labels"] for b in batch]
        batch_labels = self.tokenizer.pad(
            [{"input_ids": l} for l in labels],
            return_tensors="pt"
        )

        labels = batch_labels["input_ids"]
        labels = labels.masked_fill(batch_labels["attention_mask"].ne(1), -100)

        batch_inputs["labels"] = labels
        batch_inputs.pop("attention_mask", None)

        return batch_inputs

#compute metrics with memory management (not really effective)
def compute_metrics(pred):
    preds = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = pred.label_ids

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    del preds, labels, pred_str, label_str, pred
    torch.mps.empty_cache()
    gc.collect()

    print("\n Sample Eval Output")
    print("REF:", label_str[0])
    print("HYP:", pred_str[0])
    print("\n")

    return {"wer": wer, "cer": cer}

#main
if __name__ == "__main__":
    
    device = torch.device("mps")
    print(f"Training device set to: {device.type.upper()})")
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
    feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
    tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    model.to(device)
    model.to(torch.bfloat16) 
    model.config.use_cache = False
    sampling_rate = feature_extractor.sampling_rate

    print("Applying LoRA configuration:")
    model = get_peft_model(model, LORA_CFG)
    model.print_trainable_parameters()

    prompt_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    no_timestamps_id = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")

    new_prompt_ids = []
    for ids in prompt_ids:
        ids_list = list(ids)
        ids_list.append(no_timestamps_id)
        new_prompt_ids.append(tuple(ids_list))

    model.config.forced_decoder_ids = new_prompt_ids
    model.config.decoder_start_token_id = tokenizer.bos_token_id
    model.config.suppress_tokens = []

    print(f"Loading dataset from: {DATASET_PATH}")
    ds = load_from_disk(DATASET_PATH)
    ds_splits = ds.train_test_split(test_size=130, seed=42)
    ds = DatasetDict({"train": ds_splits["train"], "validation": ds_splits["test"]})

    print("Preprocessing dataset")
    ds_prepared = ds.map(
        prepare,
        remove_columns=ds["train"].column_names,
        num_proc=2,
        desc="Preparing dataset splits"
    )

    ds_prepared = ds_prepared.filter(
        lambda example: len(example["labels"]) > 0,
        num_proc= 2,
        desc="Filtering empty samples"
    )
    print(f"Dataset prepared. Train size: {len(ds_prepared['train'])}, Validation size: {len(ds_prepared['validation'])}")

    data_collator = WhisperDataCollator(
        tokenizer=tokenizer,
        feature_extractor=feature_extractor
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_steps=WARMUP_STEPS,

        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",

        eval_steps=EVAL_STEPS,
        logging_steps=100,
        save_steps=SAVE_STEPS,

        fp16=False,
        bf16=True,
        save_total_limit=3,
        remove_unused_columns=False,
        predict_with_generate=False,
        greater_is_better=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ds_prepared["train"],
        eval_dataset=ds_prepared["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
    processor.save_pretrained(os.path.join(OUTPUT_DIR, "final_model"))

Training device set to: MPS)
Applying LoRA configuration:
trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267
Loading dataset from: Processed_Hindi_Parquet_Dataset
Preprocessing dataset


Filtering empty samples (num_proc=2): 100%|██████████| 130/130 [00:01<00:00, 70.27 examples/s]
/opt/miniconda3/envs/whisper_fine_tuning/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Dataset prepared. Train size: 5437, Validation size: 130


Step,Training Loss,Validation Loss
